In [ ]:
import sphere_ref_lib as srl
import sphere_variables as sv

import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import xarray as xr

plt.rcParams["font.size"] = 20

#fp = srl.set_output_dir()
fp_nc = srl.set_output_dir_nc()
R, step, xs, d, Z0, R2 = srl.set_basic_params()
R2s, R2, target, apply_ref, fp, _ = sv.svv()
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = srl.get_default_param(target)
theta_arr_r2, phi_arr_r2, p0s, p2s, p_hits, ref_dirs, amp_arr, alp_arr = srl.ref_rays_count_ref(R2, xs, d, Z0, R, y=0.0, print_info=True, reflectance=False, e1=e1)
if not os.path.exists(fp_nc + "alpha_to_theta_forhist_ver2.nc"):
    R2s_make_nc = np.array(R2s)
    srl.make_nc_alpha_ts(R2s_make_nc*1000, fp_nc, fn = "alpha_to_theta_forhist_ver2.nc")

plot_colors = ["red", "darkorange", "springgreen", "mediumblue", "fuchsia"]

In [ ]:
R_norm = 1.5
xs_norm = np.arange(0,R_norm+step,step)

srl.make_nc_inc(R_norm, step, xs_norm, d, Z0, fp_nc)

In [ ]:
ds_inc = srl.load_nc_inc(1.5, fp_nc)
ds_inc = np.degrees(ds_inc)

heat_inc = srl.make_thph_2dhist(ds_inc)

stef_inc = srl.i2_the_solid_angle_field(1.5,heat_inc)

inc_counts_090 = srl.sum_phi(stef_inc, 0, 90)

In [ ]:
param_list, cor_list = [], []

for i in range(len(R2s)):

    R2 = R2s[i]

    theta_arr_r2, phi_arr_r2, p0s, p2s, p_hits, ref_dirs, amp_arr, alp_arr = srl.ref_rays_count_ref(R2, xs, d, Z0, R, y=0.0, print_info=False, reflectance=False, e1=e1)

    params = {
        "target" : target,
        "R2" : R2,
        "fp_nc" : fp_nc,
        "theta_arr_r2" : theta_arr_r2,
        "phi_arr_r2" : phi_arr_r2,
        "p0s" : p0s,
        "p2s" : p2s,
        "p_hits" : p_hits,
        "ref_dirs" : ref_dirs,
        "amp_arr" : amp_arr,
        "alp_arr" : alp_arr,
    }

    cors = [
        ("solid_angle", {}),
        #("solid_angle_field", {}),
        #("reflection_rate", {"apply_ref": "average", "fn": "alpha_to_theta_forhist_ver2.nc"}),
        #("vertical_direction", {}),
    ]
    param_list.append(params)
    cor_list.append(cors)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

R2 = 1000000

ds_all = srl.load_nc_sphere(R2, fp_nc)
heat_da = srl.make_thph_2dhist(ds_all)
fin_da = srl.steve_correction_pipeline(heat_da, params, cors, print_info=False)

fin_da = fin_da.sel(phi=slice(0,90))

fin_da.plot(ax=ax, x="theta", y="phi", cmap="inferno", vmin=0, vmax=500)

plt.savefig(fp + f"2dhist_theta_phi_R{R2}_forslide.png")

fig, ax = plt.subplots(figsize=(12,4))

th_1dcounts_090 = fin_da.sum(dim="phi") / len(fin_da["phi"])

ax.grid()
ax.set_title(f"θ 1D counts (averaged over φ=0-90 deg) R={R2}")
ax.set_xlabel("θ [deg]")
ax.set_ylabel("Ave Counts/sinθ")
ax.plot(th_1dcounts_090["theta"], th_1dcounts_090, color="mediumblue")
plt.savefig(fp + f"1dcounts_theta_R{R2}_forslide.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14,8))

ax.set_xticks(np.arange(0,181,10))
ax.grid()

test_counts = np.zeros(len(R2s))

for i in range(len(R2s)):

    R2 = R2s[i]
    params = param_list[i]
    cors = cor_list[i]

    ds_all = srl.load_nc_sphere(R2, fp_nc)

    heat_da = srl.make_thph_2dhist(ds_all)

    fin_da = srl.steve_correction_pipeline(heat_da, params, cors, print_info=False)

    phi_min, phi_max = 0.0, 90.0

    fin_da_090 = fin_da.where((fin_da["phi"]>=phi_min) & (fin_da["phi"]<=phi_max), drop=True)
    #th_1dcounts_090 = fin_da_090.sum(dim="phi")
    #th_1dcounts_090.name = "counts (sum over phi 0-90)"
    th_1dcounts_090 = fin_da_090.sum(dim="phi") / len(fin_da_090["phi"])
    th_1dcounts_090.name = "Ave Counts (sum over phi 0-90)"

    
    ax.plot(th_1dcounts_090["theta"].values, th_1dcounts_090.values / (R2**2), c=plot_colors[i], label=f"R2={R2}")
    ax.set_title(f"Ave Counts vs Theta (Counts over phi=0-90)")
    ax.set_xlabel("Theta (degrees)")
    ax.set_ylabel("Ave counts (Counts over phi 0-90)")
    #ax.set_xlim(90, 100)
    #ax.set_ylim(0,20000)

    test_counts[i] = th_1dcounts_090.values[0]

#ax.plot(inc_counts_090["theta"].values, inc_counts_090.values, c="black", ls="--", label="incident beam")

print("\ncounts ratio : ", test_counts[0] / test_counts[-1])
print("\nDistance square ratio : ", ((R2s[-1]-0.5)/(R2s[0]-0.5))**2)

plt.legend()
plt.savefig(fp + "theta_counts_090.png")
plt.show()

In [ ]:
ds_all = srl.load_nc_sphere(1.05, fp_nc)

ds_all["theta"].shape

In [ ]:
test_alth = xr.open_dataarray(fp_nc + "alpha_to_theta_forhist_ver2.nc")

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xticks(np.arange(0, 180+1, 10))
ax.grid()

for i in range(len(R2s)):
    ax.plot(test_alth.sel(H=R2s[i]*1000), label=f"R2={R2s[i]}", c=plot_colors[i])
    ax.set_title("s/c angle Alpha vs reflection angle Theta_s")
    ax.set_xlabel("Alpha (degrees)")
    ax.set_ylabel("reflection angle theta (degrees)")

plt.legend()
plt.tight_layout()
plt.savefig(fp + "alpha_vs_theta_Hver2.png")
plt.show()

In [ ]:
sum_ray, sum_field, unit_inc = srl.calc_inc_field(1.5, y_step=20)

print(unit_inc * (0.5/0.55)**2)
print(sum_ray, sum_field)

In [ ]:
print(np.radians(1))
print(np.pi/180)